# Track 2: adaptive language-subspace experiments

Run cells individually when GPU testing is authorized. No requirements-file install, automatic sweep, or embedded credentials. See `docs/KAGGLE_SUBSPACE.md` for assumptions and controls. Clone/select the experimental branch in a writable directory first, then set REPO below.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.metadata
REPO = Path('/kaggle/working/CLaS-SemEval-2027')
assert (REPO / 'subspace_generate.py').exists(), 'Clone/select the experimental branch first'
os.chdir(REPO)
for package in ['torch', 'transformers', 'accelerate', 'vllm']:
    try: print(package, importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError: print(package, 'not installed')
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True)


## CPU checks
These tests use synthetic tensors and do not load a language model.

In [ ]:
run('-m', 'unittest', 'discover', '-s', 'tests', '-v')


## Configure inputs
Use raw parallel FLORES text with verified identical sentence order. Extraction sentences must be disjoint from validation questions. The official tiny-aya model needs its own artifact; choose valid layers for that model. Use your existing Hugging Face login through Kaggle Secrets, never a literal token in a cell.

In [ ]:
MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
LANGUAGES = 'ar de en es fr ja zh ru hi ko cs el fa vi uk mt no sw th tr'.split()
PARALLEL = REPO / 'data/train'
QUESTIONS = REPO / 'data/validation'
ARTIFACT = Path('/kaggle/working/subspaces/llama.pt')
OUT = Path('/kaggle/working/subspace_runs')
LAYERS = [16, 20, 24]


## Extract (GPU)
This loads the model in a child process, which releases its GPU memory on exit. Start with 128 sentences/batch size 1 if needed; the example uses 512 sentences and batch size 4.

In [ ]:
run('extract_language_subspace.py', '--model', MODEL, '--parallel-dir', PARALLEL,
    '--languages', *LANGUAGES, '--layers', *LAYERS, '--rank', 8,
    '--num-sentences', 512, '--batch-size', 4, '--out', ARTIFACT)


## Validate the planned smoke run (no model load)

In [ ]:
COMMON = ['subspace_generate.py', '--model', MODEL, '--artifact', ARTIFACT,
          '--questions-dir', QUESTIONS, '--out', OUT, '--pairs', 'en:mt', 'en:sw',
          '--layers', 20, '--limit', 2, '--split', 'tune']
run(*COMMON, '--method', 'adaptive', '--backend', 'hf', '--dry-run')


## HF smoke comparisons (GPU)
HF defaults to final prompt position plus decoding. Each child process exits before the next starts. Inspect the two answers per pair before larger experiments.

In [ ]:
for method, strength in [('none', 0.5), ('diffmean', 5), ('fixed', 0.5), ('adaptive', 0.5)]:
    run(*COMMON, '--method', method, '--strength', strength, '--backend', 'hf', '--resume')


## Optional vLLM smoke run (GPU)
Use all-token scope. This is a different intervention scope from HF/last: compare against HF/all before comparing backends. Choose the available GPU count explicitly. Unsupported worker layouts fail loudly. If your vLLM requests serialization opt-in, review its warning in your trusted notebook; this notebook does not enable it automatically.

In [ ]:
# Uncomment when ready:
# run(*COMMON, '--method', 'adaptive', '--backend', 'vllm', '--scope', 'all',
#     '--tensor-parallel-size', 2, '--resume')


## Write a small screening plan (does not execute it)
Start with one layer and one target. Read the generated commands; do not launch a large grid blindly.

In [ ]:
PLAN = Path('/kaggle/working/screen.sh')
run('plan_subspace_sweep.py', '--artifact', ARTIFACT, '--model', MODEL,
    '--questions-dir', QUESTIONS, '--layers', 20, '--sources', 'en', 'ar', 'ru', 'hi',
    '--targets', 'mt', '--limit', 8, '--out', OUT, '--plan', PLAN)
print(PLAN.read_text())


## Evaluate after generation exits
Choose exact run directories printed above. Keep the judge/LID settings identical. Run LFS-only first only as a smoke check; obtain relevance before interpreting LSS. Compare identical pair sets and question IDs. Tune on the tune split, then freeze settings and run holdout once. Omit `--pairs` for all 100 Track 2 pairs, and remove `--limit` for full selected splits.

In [ ]:
# Fill only when ready:
# RUN_DIRECTORY = OUT / 'adaptive_hf_<printed fingerprint>'
# run('evaluate.py', '--gen-dir', RUN_DIRECTORY)
# run('compare_steering_runs.py', CONTROL_DIRECTORY / 'scores.json', RUN_DIRECTORY / 'scores.json')
